# Two models in closed form

*A companion to* The Failure That Did Not Happen.

This notebook runs the two closed-form models that the Element glosses in Appendix B and the companion states as Model S and Model A. Both are born from the Element's account but are not its subject. No claim rests on them, and a wrong rate in either leaves every claim standing.

Here the rates are given, so every curve is exact. A reading of a real record would fit the same forms and estimate the rates (Section 10.4 of the Element).

Every function used here lives in `../element-figures/compute_sieve_chain.py` and `../element-figures/compute_adequacy.py`, where each model's rules are stated at the head. The notebook draws them; it computes nothing of its own. Move a slider and the curve redraws.

In [ ]:
# In Colab the notebook arrives alone: fetch the folder it draws from and work beside its scripts. Elsewhere this does nothing.
import os, subprocess, sys

if "google.colab" in sys.modules:
    if not os.path.isdir("companion"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/davidtswanson903/failure-that-did-not-happen-companion", "companion"], check=True)
    os.chdir("companion/notebook")

In [ ]:
import sys, itertools
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, IntSlider, Dropdown, Checkbox, SelectMultiple, fixed

here = Path.cwd()
scripts = next(p for p in [here.parent / "element-figures", here / "element-figures", here]
               if (p / "compute_sieve_chain.py").exists())
sys.path.insert(0, str(scripts))
import compute_sieve_chain as sieve
import compute_adequacy as adeq

U_REF, V_REF = sieve.calibrate()
print(f"Reference rates: tight year {sieve.P}, holder leaves {sieve.Q}, "
      f"cut if unheld {U_REF:.3f}, cut if held {V_REF:.3f}")

## Part 1. The sieve as a survival model

Each capacity is held, unheld or cut. Every year is tight with chance *p*. A tight year cuts a held capacity with chance *v* and an unheld one with chance *u*. A held capacity that survives the year loses its holder with chance *q*. Cut is final. The reference rates reproduce the two endpoints Section 3.2 prints: after 30 years, 17 per cent of unheld capacities remain and 58 per cent of held ones.

### Survival

The shares still kept follow in closed form (S.1):

$$S_U(t) = b^{t}, \qquad S_H(t) = a^{t} + \frac{(1-h_1)\,q}{a-b}\left(a^{t} - b^{t}\right),$$

where $h_0 = pu$, $h_1 = pv$, $a = (1-h_1)(1-q)$ and $b = 1-h_0$. Tick the box to lay a simulation of the same rules over the curves.

In [ ]:
def show_survival(p=sieve.P, q=sieve.Q, u=U_REF, v=V_REF, simulate=False):
    r = sieve.rates(p, q, u, v)
    t = np.arange(0, 61)
    s_u = [sieve.survival(x, r)[0] for x in t]
    s_h = [sieve.survival(x, r)[1] for x in t]
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(t, s_h, color="#2f5d8a", lw=2, label="held at the start")
    ax.plot(t, s_u, color="#b5651d", lw=2, label="unheld")
    if simulate:
        sim = sieve.monte_carlo_independent_years(p, q, u, v, 60, 20000, 1)
        ax.plot(t, sim["held"], color="#2f5d8a", ls=":", lw=1)
        ax.plot(t, sim["unheld"], color="#b5651d", ls=":", lw=1)
    ax.set_xlabel("years"); ax.set_ylabel("share still kept"); ax.set_ylim(0, 1.02)
    ax.grid(color="#e6e6e6"); ax.legend(frameon=False)
    life_u, life_h = sieve.lifespan(r)
    ax.set_title(f"half-lives {sieve.half_life(r, 'unheld'):.1f} and {sieve.half_life(r, 'held'):.1f} years; "
                 f"mean years kept {life_u:.1f} and {life_h:.1f}", fontsize=9)
    plt.show()

interact(show_survival,
         p=FloatSlider(value=sieve.P, min=0.05, max=1, step=0.05, description="tight year"),
         q=FloatSlider(value=sieve.Q, min=0, max=0.2, step=0.005, readout_format=".3f", description="holder leaves"),
         u=FloatSlider(value=U_REF, min=0, max=0.6, step=0.005, readout_format=".3f", description="cut if unheld"),
         v=FloatSlider(value=V_REF, min=0, max=0.6, step=0.005, readout_format=".3f", description="cut if held"),
         simulate=Checkbox(value=False, description="simulate"));

### Without units

Measure time in units of $1/h_0$, the unheld capacity's mean wait for the cut. Two ratios are then left: $\kappa = q/h_0$, how fast holders leave against the unheld hazard, and $\varepsilon = h_1/h_0 = v/u$, how much holding protects. Every curve's shape depends on these two alone. A held capacity outlasts an unheld one by a factor of $(1+\kappa)/(\kappa+\varepsilon)$ in mean lifespan.

$$S_U(\tau) = e^{-\tau}, \qquad S_H(\tau) = e^{-(\kappa+\varepsilon)\tau} + \frac{\kappa}{1-\kappa-\varepsilon}\left(e^{-(\kappa+\varepsilon)\tau} - e^{-\tau}\right)$$

In [ ]:
def show_dimensionless(kappa=0.35, epsilon=0.16):
    tau = np.linspace(0, 6, 301)
    s_u = np.exp(-tau)
    g = kappa + epsilon
    if abs(1 - g) > 1e-9:
        s_h = np.exp(-g * tau) + kappa / (1 - g) * (np.exp(-g * tau) - np.exp(-tau))
    else:
        s_h = np.exp(-tau) * (1 + kappa * tau)
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(tau, s_h, color="#2f5d8a", lw=2, label="held at the start")
    ax.plot(tau, s_u, color="#b5651d", lw=2, label="unheld")
    ax.set_xlabel("time, in units of the unheld mean wait"); ax.set_ylabel("share still kept")
    ax.set_ylim(0, 1.02); ax.grid(color="#e6e6e6"); ax.legend(frameon=False)
    ax.set_title(f"held capacities last {(1 + kappa) / (kappa + epsilon):.2f} times as long", fontsize=9)
    plt.show()

interact(show_dimensionless,
         kappa=FloatSlider(value=0.35, min=0.01, max=3, step=0.01, description="κ"),
         epsilon=FloatSlider(value=0.16, min=0, max=1, step=0.01, description="ε"));

### The jump at departure

For one capacity, the yearly chance of being cut steps from $h_1$ to $h_0$ in the year after its holder leaves, a factor of $u/v$. This is the model's sharpest test, because it can be looked for around the departures of real sponsors and officials. Choose the year the holder leaves.

In [ ]:
def show_jump(leaves=12, p=sieve.P, u=U_REF, v=V_REF):
    years = np.arange(0, 31)
    hazard = np.where(years < leaves, p * v, p * u)
    fig, ax = plt.subplots(figsize=(7, 3.2))
    ax.step(years, hazard, where="post", color="#2f5d8a", lw=2)
    ax.axvline(leaves, color="#9a9a9a", ls="--", lw=1)
    ax.set_xlabel("year"); ax.set_ylabel("chance of being cut that year")
    ax.set_ylim(0, max(p * u, p * v) * 1.3); ax.grid(color="#e6e6e6")
    ax.set_title(f"the holder leaves in year {leaves}: the yearly risk rises "
                 f"{(u / v if v else float('inf')):.1f} times", fontsize=9)
    plt.show()

interact(show_jump, leaves=IntSlider(value=12, min=1, max=29, description="holder leaves"),
         p=fixed(sieve.P), u=fixed(U_REF), v=fixed(V_REF));

### Survivors' worth

Worth enters only through a signal the budget office registers (Claim 6). Here the chance of cutting an unheld capacity in a tight year falls as the signal rises, and the signal moves with worth as closely as the slider says. With a signal that does not move with worth, the capacities that survive are worth exactly what all of them were worth at the start (S.4).

In [ ]:
def show_worth(rho=0.5, years=30):
    direct, _, base = sieve.survivors_worth(rho, years, sieve.P, U_REF)
    rhos = np.linspace(0, 0.95, 20)
    curve = [sieve.survivors_worth(x, years, sieve.P, U_REF)[0] for x in rhos]
    fig, ax = plt.subplots(figsize=(7, 3.6))
    ax.plot(rhos, curve, color="#2f5d8a", lw=2)
    ax.axhline(base, color="#9a9a9a", ls="--", lw=1)
    ax.plot([rho], [direct], "o", color="#b5651d")
    ax.set_xlabel("how closely the registered signal moves with worth")
    ax.set_ylabel(f"mean worth of survivors, year {years}"); ax.grid(color="#e6e6e6")
    ax.set_title(f"survivors' mean worth {direct:.3f}, against {base:.3f} for all at the start", fontsize=9)
    plt.show()

interact(show_worth, rho=FloatSlider(value=0.5, min=0, max=0.95, step=0.05, description="moves with"),
         years=IntSlider(value=30, min=1, max=60, description="years"));

### One city

The closed forms give the chances for one capacity. The capacities of one city meet the same tight years, so a city's shares scatter around the closed form from one run to the next, and its cuts come in waves. Change the seed to run another city.

In [ ]:
def show_city(seed=1):
    r = sieve.rates(sieve.P, sieve.Q, U_REF, V_REF)
    run = sieve.one_city_run(sieve.P, sieve.Q, U_REF, V_REF, 40, 2000, seed)
    t = np.arange(0, 41)
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(t, [sieve.survival(x, r)[1] for x in t], color="#2f5d8a", lw=2, label="held, closed form")
    ax.plot(t, [sieve.survival(x, r)[0] for x in t], color="#b5651d", lw=2, label="unheld, closed form")
    ax.step(t, run["held"], where="post", color="#2f5d8a", lw=1, ls=":", label="held, one city")
    ax.step(t, run["unheld"], where="post", color="#b5651d", lw=1, ls=":", label="unheld, one city")
    ax.set_xlabel("years"); ax.set_ylabel("share still kept"); ax.set_ylim(0, 1.02)
    ax.grid(color="#e6e6e6"); ax.legend(frameon=False, fontsize=8)
    plt.show()

interact(show_city, seed=IntSlider(value=1, min=1, max=50, description="city"));

### Exercise

Find rates at which holding does not lengthen a capacity's life at all, and say why.

<details><summary>Answer</summary>

Set the chance a tight year cuts a held capacity equal to the chance it cuts an unheld one, <i>v</i> = <i>u</i>. Then losing a holder changes nothing, and the two curves coincide. Holding lengthens life only because the holder makes the cut less likely; where the holder's loss does not reach the decision, the holding does no work.
</details>

## Part 2. Adequacy for a purpose

A register's adequacy for a purpose is the share of the states that occur at which what the register reads settles the purpose. It is 1 exactly where the register carries the purpose. Two states the register cannot tell apart but the purpose can are a *confusion*. The measure is Pawlak's degree of dependency from rough-set theory, taken over the states that occur; the name is Parker's.

### The bridge, live

The bridge of Section 2.1 has three yes-or-no facts: a crack, funding, corrosion inside the cable. Pick a register and a purpose.

In [ ]:
S, sound, REGISTERS, _, _, _ = adeq.bridge()
PURPOSES = {
    "sound (no crack, no corrosion)": sound,
    "corroded": lambda x: x[2],
    "cracked": lambda x: x[0],
    "funded": lambda x: x[1],
}
NAMES = {"inspection": "inspection unit (crack)", "budget": "budget office (funding)",
         "both_offices": "both offices together", "ultrasound": "inspection with ultrasound"}

def show_bridge(register="inspection", purpose="sound (no crack, no corrosion)"):
    c, q = REGISTERS[register], PURPOSES[purpose]
    R = adeq.region(c, q, S)
    conf = adeq.confusions(c, q, S)
    pairs = list(itertools.combinations(S, 2))
    reads_q = lambda x: (c(x), q(x))
    print(f"{NAMES[register]}, for '{purpose}'")
    print(f"  adequacy {adeq.adequacy(c, q, S):.3f}: settled at {len(R)} of {len(S)} states")
    print(f"  confusions: {len(conf)}")
    print(f"  cheapest refinement that makes it adequate costs {adeq.price(c, reads_q, pairs)} pairs")
    print()
    print("  crack funding corrosion | purpose | settled")
    for x in S:
        print(f"    {x[0]}      {x[1]}        {x[2]}      |    {q(x)}    |   {'yes' if x in R else 'no'}")

interact(show_bridge, register=Dropdown(options=[(v, k) for k, v in NAMES.items()], value="inspection"),
         purpose=Dropdown(options=list(PURPOSES), value="sound (no crack, no corrosion)"));

Ultrasound makes the inspection unit tell apart 8 pairs it did not before, the price Section 2.5 counts. Adequacy for soundness needs only 4 of them, the inspection unit's confusions. The other 4 buy corrosion on bridges already cracked, which soundness does not need.

### Borrowing

Age stands in for corrosion only over the cables the city actually has (Claim 6). Choose which kinds of cable are in service and watch the adequacy of age for corrosion.

In [ ]:
KINDS = {"new, sound": (0, 0), "new, corroded": (0, 1), "old, sound": (1, 0), "old, corroded": (1, 1)}

def show_borrowing(in_service=("new, sound", "old, corroded")):
    X = [KINDS[k] for k in in_service]
    if not X:
        print("Choose at least one kind of cable."); return
    age, corroded = (lambda x: x[0]), (lambda x: x[1])
    print(f"cables in service: {', '.join(in_service)}")
    print(f"adequacy of age for corrosion: {adeq.adequacy(age, corroded, X):.3f}")
    for k in in_service:
        print(f"  {k:14s} settled: {'yes' if KINDS[k] in adeq.region(age, corroded, X) else 'no'}")

interact(show_borrowing, in_service=SelectMultiple(options=list(KINDS), value=("new, sound", "old, corroded"),
                                                    rows=4, description="in service"));

Widening the world can only take states out of the region (A.6). The *share* can still rise, because new states may arrive already settled: add "old, corroded" alone to "new, sound" and "new, corroded", and the adequacy goes from 0 to 1/3.

### Pooling

An arrangement's register is its parties' registers read together. It is at least as adequate as its best party, and a state falls short of it exactly where a pair of the between holds it and the purpose separates that pair (A.5). Choose which facts each party reads.

In [ ]:
FACTS = {"crack": 0, "funding": 1, "corrosion": 2}

def show_pooling(first=("crack",), second=("funding",), purpose="sound (no crack, no corrosion)"):
    q = PURPOSES[purpose]
    c1 = lambda x: tuple(x[FACTS[f]] for f in first)
    c2 = lambda x: tuple(x[FACTS[f]] for f in second)
    J = adeq.joint(c1, c2)
    btw = adeq.between([c1, c2], S)
    sep = [pair for pair in btw if q(pair[0]) != q(pair[1])]
    print(f"first party reads {', '.join(first) or 'nothing'}: adequacy {adeq.adequacy(c1, q, S):.3f}")
    print(f"second party reads {', '.join(second) or 'nothing'}: adequacy {adeq.adequacy(c2, q, S):.3f}")
    print(f"together: adequacy {adeq.adequacy(J, q, S):.3f}")
    print(f"pairs between them: {len(btw)}, of which the purpose separates {len(sep)}")

interact(show_pooling,
         first=SelectMultiple(options=list(FACTS), value=("crack",), rows=3, description="first reads"),
         second=SelectMultiple(options=list(FACTS), value=("funding",), rows=3, description="second reads"),
         purpose=Dropdown(options=list(PURPOSES), value="sound (no crack, no corrosion)"));

### Over time

While a correction is kept, adequacy stays at 1. Once it is cut, the world moves and adequacy falls by a factor $1 - m$ each year. The correction is a capacity like any other, held or unheld, with the sieve's rates. The expected adequacy follows in closed form ($E_0$, $E_U$, $E_H$ in the companion). The dashed line is the world's pace alone, with no correction at all.

In [ ]:
def show_over_time(m=adeq.M, p=sieve.P, q=sieve.Q, u=U_REF, v=V_REF, simulate=False):
    r = sieve.rates(p, q, u, v)
    t = np.arange(0, 41)
    curves = {start: [adeq.expected_adequacy(x, m, start, r) for x in t] for start in ("none", "unheld", "held")}
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(t, curves["held"], color="#2f5d8a", lw=2, label="correction held")
    ax.plot(t, curves["unheld"], color="#b5651d", lw=2, label="correction unheld")
    ax.plot(t, curves["none"], color="#9a9a9a", lw=1.5, ls="--", label="no correction")
    if simulate:
        sim = adeq.adequacy_monte_carlo(m, 40, r, 20000, 1)
        ax.plot([40, 40], [sim["held"], sim["unheld"]], "o", color="#333333", ms=4, label="simulated, year 40")
    ax.set_xlabel("years"); ax.set_ylabel("expected adequacy"); ax.set_ylim(0, 1.02)
    ax.grid(color="#e6e6e6"); ax.legend(frameon=False, fontsize=8)
    ax.set_title(f"year 40: {curves['none'][-1]:.2f} with no correction, {curves['unheld'][-1]:.2f} unheld, "
                 f"{curves['held'][-1]:.2f} held", fontsize=9)
    plt.show()

interact(show_over_time,
         m=FloatSlider(value=adeq.M, min=0.0, max=0.15, step=0.005, readout_format=".3f", description="world moves"),
         p=FloatSlider(value=sieve.P, min=0.05, max=1, step=0.05, description="tight year"),
         q=FloatSlider(value=sieve.Q, min=0, max=0.2, step=0.005, readout_format=".3f", description="holder leaves"),
         u=FloatSlider(value=U_REF, min=0, max=0.6, step=0.005, readout_format=".3f", description="cut if unheld"),
         v=FloatSlider(value=V_REF, min=0, max=0.6, step=0.005, readout_format=".3f", description="cut if held"),
         simulate=Checkbox(value=False, description="simulate"));

### Exercises

1. Find a register that is adequate for a purpose on a narrow world and not on a wide one.
2. Find the cheapest refinement that makes the bridge's inspection unit adequate for soundness.

<details><summary>Answers</summary>

1. Age for corrosion, with only new sound cables and old corroded ones in service (adequacy 1); put every kind in service and it falls to 0.
2. Let the inspection unit also read soundness itself, or corrosion on uncracked bridges only. Either tells apart exactly its 4 confusions, the pairs of uncracked bridges that differ in corrosion. Full ultrasound tells apart 8 pairs; the other 4 are on cracked bridges, which are unsound either way.
</details>

## Part 3. The checks, as cells

These run every assertion in both scripts and rewrite their numbers files, which come out the same each time because every seed is fixed. Each takes a few seconds.

In [ ]:
sieve.main()
print("compute_sieve_chain.py: every check passed")

In [ ]:
adeq.main()
print("compute_adequacy.py: every check passed")